In [1]:
###################################
# IMPORT
###################################


from polars import col, concat, lit, when
import polars as pl

import inspect
import sys; sys.path.append('.'); 
from scripts.hdb_helpers import sample_hdb
import scripts.hdb_helpers as dc


# SAMPLED = 'on'
SAMPLED = None

In [2]:
###################################
# DATA LOAD
###################################
hdbdata = pl.read_parquet('datalake/hdb/cleaned/hdbdata')
hdbdata.glimpse()

Rows: 90943
Columns: 13
$ month               <str> '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01'
$ town                <str> 'BUKIT MERAH', 'QUEENSTOWN', 'KALLANG/WHAMPOA', 'BUKIT MERAH', 'PASIR RIS', 'BUKIT MERAH', 'KALLANG/WHAMPOA', 'MARINE PARADE', 'MARINE PARADE', 'CENTRAL AREA'
$ flat_type           <str> '5 ROOM', 'EXECUTIVE', 'EXECUTIVE', '5 ROOM', 'EXECUTIVE', '5 ROOM', '3 ROOM', '5 ROOM', '5 ROOM', '5 ROOM'
$ block               <str> '45', '150', '46', '23', '130', '2C', '57', '73', '72', '533'
$ street_name         <str> 'LENGKOK BAHRU', 'MEI LING ST', 'BENDEMEER RD', 'JLN MEMBINA', 'PASIR RIS ST 11', 'BOON TIONG RD', "JLN MA'MOR", 'MARINE DR', 'MARINE DR', 'UPP CROSS ST'
$ storey_range        <str> '01 TO 03', '07 TO 09', '16 TO 18', '13 TO 15', '07 TO 09', '13 TO 15', '01 TO 03', '22 TO 24', '22 TO 24', '10 TO 12'
$ floor_area_sqm      <f64> 139.0, 147.0, 146.0, 110.0, 188.0, 115.0, 128.0, 120.0, 117.0, 120.0


In [3]:
# User may review the source code for helper function random sampling the hdb dataset
print(inspect.getsource(sample_hdb))

def sample_hdb(hdb_df, N_ROWS, SAMPLE_SEED):
    '''
    For sampling hdb data to N_ROWS

    '''
    # take full data why not
    import polars as pl
    core_cols = [c for c in hdb_df.columns if c != 'remaining_lease']
    dataload_nonulls = hdb_df.drop_nulls(subset=core_cols)

    # sampling portion
    SAMPLE_SEED = 1   # change to 2, 3, 4, 5 for different samples
    N_PER_TABLE= round(N_ROWS/3)

    df_sample = (
        dataload_nonulls
        .group_by('tabling_version', maintain_order=True)
        .map_groups(lambda g: g.sample(n=min(N_PER_TABLE, g.height), seed=SAMPLE_SEED))
    )
    return(df_sample)

    hdbdata = df_sample



In [4]:
#### SAMPLING FOR FIRST EASE OF Validaton during dev ###
if SAMPLED == 'on': 
    hdbdata_v1 = sample_hdb(hdbdata, N_ROWS=15, SAMPLE_SEED=4)
else:
    hdbdata_v1 = hdbdata

###################################################

hdbdata_v1.glimpse()

Rows: 90943
Columns: 13
$ month               <str> '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01'
$ town                <str> 'BUKIT MERAH', 'QUEENSTOWN', 'KALLANG/WHAMPOA', 'BUKIT MERAH', 'PASIR RIS', 'BUKIT MERAH', 'KALLANG/WHAMPOA', 'MARINE PARADE', 'MARINE PARADE', 'CENTRAL AREA'
$ flat_type           <str> '5 ROOM', 'EXECUTIVE', 'EXECUTIVE', '5 ROOM', 'EXECUTIVE', '5 ROOM', '3 ROOM', '5 ROOM', '5 ROOM', '5 ROOM'
$ block               <str> '45', '150', '46', '23', '130', '2C', '57', '73', '72', '533'
$ street_name         <str> 'LENGKOK BAHRU', 'MEI LING ST', 'BENDEMEER RD', 'JLN MEMBINA', 'PASIR RIS ST 11', 'BOON TIONG RD', "JLN MA'MOR", 'MARINE DR', 'MARINE DR', 'UPP CROSS ST'
$ storey_range        <str> '01 TO 03', '07 TO 09', '16 TO 18', '13 TO 15', '07 TO 09', '13 TO 15', '01 TO 03', '22 TO 24', '22 TO 24', '10 TO 12'
$ floor_area_sqm      <f64> 139.0, 147.0, 146.0, 110.0, 188.0, 115.0, 128.0, 120.0, 117.0, 120.0


In [5]:
'''
REQUIREMENT: Taking the cleaned data from the previous step, create a new column called “Resale
Identifier”
. The “Resale Identifier” is derived from:
- First character is “S”.

- Next 3 digits is the first 3 digits of the block column, after removing any characters. In
the event the block column has less than 3 digits, prepend with the appropriate number
of zeros in front. (E.g. if block number is “19”, digits will be “019”)

- Next 2 digits is derived by: Taking the 1st and 2nd digit of the average resale price, group
by year-month, town and flat_type. For instance, if the average Ang Mo Kio, 2 Room
Flats in Jan 2017 is $230000, the 2 digits is “23”.

- The last two digits is the month of the current entry (e.g. if the month is 2012-01, the
last two digits is 01).

- The final character is the first character of the town. E.g. Ang Mo Kio, the character is
“A”.
'''

'\nREQUIREMENT: Taking the cleaned data from the previous step, create a new column called “Resale\nIdentifier”\n. The “Resale Identifier” is derived from:\n- First character is “S”.\n\n- Next 3 digits is the first 3 digits of the block column, after removing any characters. In\nthe event the block column has less than 3 digits, prepend with the appropriate number\nof zeros in front. (E.g. if block number is “19”, digits will be “019”)\n\n- Next 2 digits is derived by: Taking the 1st and 2nd digit of the average resale price, group\nby year-month, town and flat_type. For instance, if the average Ang Mo Kio, 2 Room\nFlats in Jan 2017 is $230000, the 2 digits is “23”.\n\n- The last two digits is the month of the current entry (e.g. if the month is 2012-01, the\nlast two digits is 01).\n\n- The final character is the first character of the town. E.g. Ang Mo Kio, the character is\n“A”.\n'

In [6]:
hdbdata_v2_ided = (
    hdbdata_v1
    .with_columns(
        avg_price_group = col('resale_price').mean().over(['month', 'town', 'flat_type']).round(0).cast(pl.Int64)
    )
    .with_columns(
        block_digits = col('block').str.replace_all(r'[^0-9]', '').str.zfill(3).str.slice(0, 3),
        avg_price_digits = col('avg_price_group').cast(pl.String).str.slice(0, 2),
        month_digits = col('month').str.slice(5, 2),
        town_initial = col('town').str.slice(0, 1),
    )
    .with_columns(
        resale_identifier_validate = pl.format('S-{}-{}-{}-{}', col('block_digits'), col('avg_price_digits'), col('month_digits'), col('town_initial')),
        resale_identifier = pl.format('S{}{}{}{}', col('block_digits'), col('avg_price_digits'), col('month_digits'), col('town_initial')),
    )
)

In [7]:
# validate
# suggestion: run this repeatedly to re-randomized and look at the resale_identifier for satsifactory verification
validate = (hdbdata_v2_ided
            .select('record_id', 'month', 'town', 'flat_type', 'block', 'resale_price',
                    'avg_price_group', 'block_digits', 'avg_price_digits', 'month_digits', 'town_initial',
                    'resale_identifier_validate', 'resale_identifier')
            .sort('month', 'town', 'flat_type'
            )
)

# sample one random (month, town, flat_type) group to manually spot-check the avg_price_group math
VALIDATE_GROUP_SEED = 4

random_group = (
    validate
    .select('month', 'town', 'flat_type')
    .unique(maintain_order=True)
    .sample(n=1, seed=None)
)
print(random_group)

validate_group = validate.join(random_group, on=['month', 'town', 'flat_type'], how='inner')
dc.showall(validate_group)



shape: (1, 3)
┌─────────┬───────┬───────────┐
│ month   ┆ town  ┆ flat_type │
│ ---     ┆ ---   ┆ ---       │
│ str     ┆ str   ┆ str       │
╞═════════╪═══════╪═══════════╡
│ 2013-07 ┆ BEDOK ┆ EXECUTIVE │
└─────────┴───────┴───────────┘
shape: (4, 13)
┌────────────┬─────────┬───────┬───────────┬───────┬──────────────┬─────────────────┬──────────────┬──────────────────┬──────────────┬──────────────┬────────────────────────────┬───────────────────┐
│ record_id  ┆ month   ┆ town  ┆ flat_type ┆ block ┆ resale_price ┆ avg_price_group ┆ block_digits ┆ avg_price_digits ┆ month_digits ┆ town_initial ┆ resale_identifier_validate ┆ resale_identifier │
│ ---        ┆ ---     ┆ ---   ┆ ---       ┆ ---   ┆ ---          ┆ ---             ┆ ---          ┆ ---              ┆ ---          ┆ ---          ┆ ---                        ┆ ---               │
│ str        ┆ str     ┆ str   ┆ str       ┆ str   ┆ f64          ┆ i64             ┆ str          ┆ str              ┆ str          ┆ str          ┆ 

In [8]:
key_identifier = hdbdata_v2_ided.select('record_id', 'resale_identifier')

'''
REQUIREMENT: If there are any duplicate records, take the higher price and discard the lower price one.
'''

'''
A note about the resale identifier, it retains only unique information from 
year-month, town and flat_type and block number.
it excludes informaton from: street_name, storey_range, and lease_commence_year.

Hence, implementing the dedup on identifier is a lossier key than the earlier composite-key.
It intends to take one record as all of the storey_range of the block 
maybe taking the each block's highest sale price that month as the 'Y value'

# actually i am doubtful about why do this at all. the id is lossy.
# could it be that it is a repeat information of the above.

Resolution: Assumed that this is not a repeat sentence to be applied to composite key, and the 
ask is to crete a dataset with 1 resale_identifier as 1 record. 
'''

"\nA note about the resale identifier, it retains only unique information from \nyear-month, town and flat_type and block number.\nit excludes informaton from: street_name, storey_range, and lease_commence_year.\n\nHence, implementing the dedup on identifier is a lossier key than the earlier composite-key.\nIt intends to take one record as all of the storey_range of the block \nmaybe taking the each block's highest sale price that month as the 'Y value'\n\n# actually i am doubtful about why do this at all. the id is lossy.\n# could it be that it is a repeat information of the above.\n\nResolution: Assumed that this is not a repeat sentence to be applied to composite key, and the \nask is to crete a dataset with 1 resale_identifier as 1 record. \n"

In [9]:
# Do not use price==max(price)
# hdbdata_id_keyed = (
#     hdbdata_id
#     .with_columns(max_price_in_id = col('resale_price').max().over('resale_identifier'))
# )
#
# hdbdata_id_dedup = (
#     hdbdata_id_keyed
#     .filter(col('resale_price') == col('max_price_in_id'))
#     .drop('max_price_in_id')
# )
#
# failed_id_duplicates = (
#     hdbdata_id_keyed
#     .join(hdbdata_id_dedup, on='record_id', how='anti')
#     .drop('max_price_in_id')
# )

In [10]:
# Resolution: Alternate to tie breaker filter row_number == 1, over composite_key
# sort by resale_price descending, then record_id ascending as a deterministic
# tiebreak, then keep only the first row encountered per identifier - guarantees
# exactly one row survives per identifier, even on a genuine price tie.


key_identifier_plus_price = key_identifier.join(hdbdata_v1.select('record_id', 'resale_price'), on='record_id', how='left')
dc.unicity(key_identifier_plus_price, 'resale_identifier')
# failed

victims will be returned in global as uvictims
victims are returned in global as uvictims.
multiplicity pcnt
77254
shape: (10, 3)
┌─────────┬───────┬──────────┐
│ nb_rows ┆ count ┆ pcnt     │
│ ---     ┆ ---   ┆ ---      │
│ u32     ┆ u32   ┆ f64      │
╞═════════╪═══════╪══════════╡
│ 1       ┆ 66305 ┆ 0.858273 │
│ 2       ┆ 8973  ┆ 0.116149 │
│ 3       ┆ 1485  ┆ 0.019222 │
│ 4       ┆ 328   ┆ 0.004246 │
│ 5       ┆ 104   ┆ 0.001346 │
│ 6       ┆ 31    ┆ 0.000401 │
│ 7       ┆ 16    ┆ 0.000207 │
│ 9       ┆ 5     ┆ 0.000065 │
│ 8       ┆ 5     ┆ 0.000065 │
│ 11      ┆ 2     ┆ 0.000026 │
└─────────┴───────┴──────────┘
unicity ended


In [11]:
key_identifier_dedup = (
    key_identifier_plus_price
    .sort(['resale_price', 'record_id'], descending=[True, False])
    .unique(subset='resale_identifier', keep='first', maintain_order=True)
    .drop('resale_price')
)

In [12]:
# validate
dc.unicity(key_identifier_dedup, 'resale_identifier')
# passed

victims will be returned in global as uvictims
good!: multiplicity passed!:
unicity ended


In [13]:
# validate
dc.dfratio(key_identifier_plus_price, key_identifier_dedup)
# 26 percent dropped

before count: 90943
after count: 77254
ratiodelta: 0.849
ratio_delta returned as global variable for exception raising


In [14]:
# filter hdbdata_v1 to only dedup records, attaching the resale_identifier value in key_identifier
hdbdata_v3_deduped = hdbdata_v1.join(key_identifier_dedup, on='record_id', how='right')


hdbdata_v3_failed_resale_identifier = (
    hdbdata_v1
    .join(hdbdata_v3_deduped, on='record_id', how='anti')
)

print(f"kept: {hdbdata_v3_deduped.shape[0]:,} rows")
print(f"discarded to failed (lower-priced identifier duplicates): {hdbdata_v3_failed_resale_identifier.shape[0]:,} rows")



kept: 77,254 rows
discarded to failed (lower-priced identifier duplicates): 13,689 rows


In [15]:
#################
### DATA WRITE to Datalake
#################
if SAMPLED == 'on':
    print('on sampled data, no WRITE')
    print('break')
else:
    OUTPUT = hdbdata_v3_failed_resale_identifier
    OUTPUT.write_parquet('datalake/hdb/failed/hdbdata_duplicate_resale_identifier_discard', partition_by=['tabling_version'], mkdir=True)

    OUTPUT = hdbdata_v3_deduped
    OUTPUT.glimpse()
    OUTPUT.write_parquet('datalake/hdb/transformed/hdbdata', partition_by=['tabling_version', 'month'], mkdir=True)
    print('written parquet')



Rows: 77254
Columns: 14
$ month               <str> '2016-12', '2016-09', '2016-08', '2016-11', '2014-10', '2015-11', '2016-06', '2016-08', '2016-01', '2016-05'
$ town                <str> 'KALLANG/WHAMPOA', 'CENTRAL AREA', 'KALLANG/WHAMPOA', 'CENTRAL AREA', 'BISHAN', 'CENTRAL AREA', 'CENTRAL AREA', 'CENTRAL AREA', 'CENTRAL AREA', 'CENTRAL AREA'
$ flat_type           <str> '3 ROOM', '5 ROOM', '5 ROOM', '5 ROOM', 'EXECUTIVE', '5 ROOM', '5 ROOM', '5 ROOM', '5 ROOM', '5 ROOM'
$ block               <str> '57', '1G', '8', '1B', '194', '1D', '1D', '1B', '1B', '1D'
$ street_name         <str> "JLN MA'MOR", 'CANTONMENT RD', 'BOON KENG RD', 'CANTONMENT RD', 'BISHAN ST 13', 'CANTONMENT RD', 'CANTONMENT RD', 'CANTONMENT RD', 'CANTONMENT RD', 'CANTONMENT RD'
$ storey_range        <str> '01 TO 03', '43 TO 45', '28 TO 30', '31 TO 33', '22 TO 24', '43 TO 45', '34 TO 36', '31 TO 33', '28 TO 30', '19 TO 21'
$ floor_area_sqm      <f64> 259.0, 106.0, 119.0, 106.0, 150.0, 107.0, 107.0, 105.0, 108.0, 106.0